# Unidad 2 · Colab 3 de 3
## Archivos estructurados y el procesador de transacciones

**Objetivos de este notebook**

- Leer y escribir archivos en JSON, CSV, YAML y Parquet.
- Elegir el formato correcto según el caso de uso.
- Aplicar todo lo de esta unidad (excepciones, módulos, formatos de archivo) en un módulo procesador de transacciones con validación de datos y logs de errores.

> **Nivel:** intermedio. Este notebook integra los Colab 1 y 2 de esta unidad.

---

## 1. JSON

Formato de texto, legible, ideal para configuración y APIs (ya lo usaste en la Unidad 4).

```python
import json

datos = {'id': 1, 'monto': 150.5, 'moneda': 'ARS'}

with open('transaccion.json', 'w') as f:
    json.dump(datos, f, indent=2)

with open('transaccion.json') as f:
    cargado = json.load(f)

print(cargado)
```

Documentación oficial: [módulo json](https://docs.python.org/3/library/json.html)

## 2. CSV

Formato tabular de texto plano, el más simple para intercambiar datos entre sistemas (Excel, bases de datos, etc.).

```python
import csv

filas = [{'id': 1, 'monto': 150.5, 'moneda': 'ARS'}, {'id': 2, 'monto': 40.0, 'moneda': 'USD'}]

with open('transacciones.csv', 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['id', 'monto', 'moneda'])
    writer.writeheader()
    writer.writerows(filas)

with open('transacciones.csv') as f:
    reader = csv.DictReader(f)
    for fila in reader:
        print(fila)
```

Documentación oficial: [módulo csv](https://docs.python.org/3/library/csv.html)

### Ejercicio 1 — JSON y CSV

Escribí el código para: (1) guardar una lista de 3 transacciones (diccionarios con `id`, `monto`, `moneda`) en `transacciones.json` usando `json.dump` con `indent=2`, y (2) leerla de vuelta y confirmar que son iguales a la lista original.

<details>
<summary>💡 Ver solución</summary>

```python
import json

transacciones = [
    {'id': 1, 'monto': 100.0, 'moneda': 'ARS'},
    {'id': 2, 'monto': 200.0, 'moneda': 'USD'},
    {'id': 3, 'monto': 50.0, 'moneda': 'EUR'},
]

with open('transacciones.json', 'w') as f:
    json.dump(transacciones, f, indent=2)

with open('transacciones.json') as f:
    cargadas = json.load(f)

print(cargadas == transacciones)
```

</details>

## 3. YAML

Más legible que JSON para humanos (sin llaves ni comillas obligatorias), muy usado para archivos de configuración — ya lo viste en los workflows de GitHub Actions (Unidad 6).

```python
!pip install -q pyyaml
import yaml

configuracion = {
    'monedas_permitidas': ['ARS', 'USD', 'EUR'],
    'monto_maximo': 100000,
    'logging': {'nivel': 'INFO', 'archivo': 'errores.log'},
}

with open('config.yaml', 'w') as f:
    yaml.dump(configuracion, f)

with open('config.yaml') as f:
    print(f.read())

with open('config.yaml') as f:
    cargada = yaml.safe_load(f)
print(cargada)
```

`yaml.safe_load` (en vez de `yaml.load`) evita ejecutar código arbitrario al leer un YAML de una fuente no confiable — es la opción recomendada.

Documentación oficial: [PyYAML](https://pyyaml.org/wiki/PyYAMLDocumentation)

## 4. Parquet

Un formato binario y columnar, pensado para grandes volúmenes de datos analíticos: comprime mejor que CSV/JSON y es mucho más rápido de leer cuando solo necesitás algunas columnas de una tabla enorme. Es el formato típico en pipelines de datos (Spark, data lakes).

```python
!pip install -q pandas pyarrow
import pandas as pd

df = pd.DataFrame([
    {'id': 1, 'monto': 100.0, 'moneda': 'ARS'},
    {'id': 2, 'monto': 200.0, 'moneda': 'USD'},
])

df.to_parquet('transacciones.parquet')
leido = pd.read_parquet('transacciones.parquet')
print(leido)
```

| Formato | Legible por humanos | Tamaño | Uso típico |
|---|---|---|---|
| JSON | Sí | Medio | Configuración, APIs |
| CSV | Sí | Medio | Intercambio simple, planillas |
| YAML | Sí (el más legible) | Medio | Configuración |
| Parquet | No (binario) | Chico (comprimido) | Analítica, grandes volúmenes |

Documentación oficial: [pandas.read_parquet](https://pandas.pydata.org/docs/reference/api/pandas.read_parquet.html) · [Apache Parquet](https://parquet.apache.org/docs/)

### Ejercicio 2 — Elegir el formato

Para cada caso, indicá qué formato usarías (JSON, CSV, YAML o Parquet) y por qué:

(a) El archivo de configuración de un workflow de GitHub Actions.

(b) Un dataset de 5 millones de filas de transacciones históricas para análisis.

(c) Exportar una lista de productos para que alguien la abra en Excel.

<details>
<summary>💡 Ver solución</summary>

(a) YAML: es el formato que usa GitHub Actions, y es el más legible para configuración escrita a mano.

(b) Parquet: por el volumen, la compresión y la velocidad de lectura columnar son claves.

(c) CSV: es lo que Excel abre de forma más directa y simple.

</details>

## 5. Aplicación práctica: módulo procesador de transacciones

Vamos a construir un módulo que integra todo lo visto en la Unidad 2:

- **Excepciones personalizadas** (Colab 1) para validar cada transacción.
- **Logging estructurado** de los errores encontrados, en vez de que el programa se detenga.
- **Lectura desde CSV** y **escritura de resultados a Parquet** (las transacciones válidas) y a **JSON Lines** (el log de errores).

In [ ]:
%%writefile procesador_transacciones.py
"""Modulo procesador de transacciones con validacion y logging de errores."""
import csv
import json
import logging
from dataclasses import dataclass

logging.basicConfig(
    filename='errores.log',
    level=logging.ERROR,
    format='%(asctime)s %(levelname)s %(message)s',
)
logger = logging.getLogger('procesador_transacciones')

MONEDAS_VALIDAS = {'ARS', 'USD', 'EUR'}


class MontoInvalidoError(Exception):
    """Se lanza cuando el monto de una transaccion no es valido."""


class MonedaInvalidaError(Exception):
    """Se lanza cuando la moneda de una transaccion no esta permitida."""


@dataclass
class Transaccion:
    id: int
    monto: float
    moneda: str


def validar_transaccion(t: Transaccion) -> None:
    """Valida una transaccion.

    Args:
        t: la transaccion a validar.

    Raises:
        MontoInvalidoError: si el monto no es mayor a cero.
        MonedaInvalidaError: si la moneda no esta en MONEDAS_VALIDAS.
    """
    if t.monto <= 0:
        raise MontoInvalidoError(f'Transaccion {t.id}: monto {t.monto} invalido')
    if t.moneda not in MONEDAS_VALIDAS:
        raise MonedaInvalidaError(f'Transaccion {t.id}: moneda {t.moneda} invalida')


def procesar_csv(ruta_entrada: str) -> list:
    """Lee transacciones desde un CSV, valida cada una y loguea los errores.

    Args:
        ruta_entrada: ruta al archivo CSV con columnas id, monto, moneda.

    Returns:
        La lista de transacciones validas como objetos Transaccion.
    """
    validas = []
    with open(ruta_entrada) as f:
        reader = csv.DictReader(f)
        for fila in reader:
            t = Transaccion(id=int(fila['id']), monto=float(fila['monto']), moneda=fila['moneda'])
            try:
                validar_transaccion(t)
                validas.append(t)
            except (MontoInvalidoError, MonedaInvalidaError) as e:
                logger.error(str(e))
    return validas

In [ ]:
csv_prueba = '''id,monto,moneda
1,100.0,ARS
2,-50.0,USD
3,75.0,GBP
4,300.0,EUR
'''

with open('transacciones_entrada.csv', 'w') as f:
    f.write(csv_prueba)

import procesador_transacciones as pt

validas = pt.procesar_csv('transacciones_entrada.csv')
print(f'{len(validas)} transacciones validas de 4')
for t in validas:
    print(t)

with open('errores.log') as f:
    print(f.read())

### Ejercicio 3 — Guardar las transacciones válidas

Agregá al módulo (o escribí acá directamente) una función `guardar_parquet(transacciones: list, ruta: str) -> None` que convierta la lista de `Transaccion` a un `DataFrame` de pandas y la guarde como Parquet.

<details>
<summary>💡 Ver solución</summary>

```python
import pandas as pd

def guardar_parquet(transacciones, ruta):
    df = pd.DataFrame([{'id': t.id, 'monto': t.monto, 'moneda': t.moneda} for t in transacciones])
    df.to_parquet(ruta)

guardar_parquet(validas, 'transacciones_validas.parquet')
print(pd.read_parquet('transacciones_validas.parquet'))
```

</details>

### Ejercicio 4 — Log en formato JSON Lines

En vez de un log de texto plano, muchos sistemas prefieren JSON Lines (un JSON por línea) porque es más fácil de procesar después. Modificá la función `validar_transaccion` (o escribí una versión nueva) para que, en vez de `logger.error(str(e))`, escriba una línea JSON a un archivo `errores.jsonl` con `{'id': t.id, 'error': str(e), 'tipo': type(e).__name__}`.

<details>
<summary>💡 Ver solución</summary>

```python
import json

def procesar_csv_v2(ruta_entrada, ruta_errores='errores.jsonl'):
    validas = []
    with open(ruta_entrada) as f, open(ruta_errores, 'w') as log:
        reader = pt.csv.DictReader(f)
        for fila in reader:
            t = pt.Transaccion(id=int(fila['id']), monto=float(fila['monto']), moneda=fila['moneda'])
            try:
                pt.validar_transaccion(t)
                validas.append(t)
            except (pt.MontoInvalidoError, pt.MonedaInvalidaError) as e:
                log.write(json.dumps({'id': t.id, 'error': str(e), 'tipo': type(e).__name__}) + '\n')
    return validas

procesar_csv_v2('transacciones_entrada.csv')
with open('errores.jsonl') as f:
    print(f.read())
```

</details>

## Mini-proyecto final: pipeline completo

Extendé `procesador_transacciones.py` para que sea un pipeline de punta a punta:

1. `procesar_csv(ruta_entrada)` — como ya está, valida y separa transacciones válidas/inválidas.
2. `guardar_parquet(transacciones, ruta)` — guarda las válidas.
3. `guardar_log_errores(errores, ruta)` — guarda los errores en JSON Lines (uno por línea).
4. `generar_reporte(validas, errores) -> dict` — devuelve un resumen: `{'total': N, 'validas': X, 'invalidas': Y, 'tasa_error': Y/N}`.
5. Un bloque `if __name__ == '__main__':` que corra las 4 funciones en orden sobre un archivo de entrada, y muestre el reporte final.

**Entregable:** el módulo `procesador_transacciones.py` completo + el archivo Parquet + el log de errores + el reporte impreso, corridos sobre al menos 10 transacciones (con al menos 3 inválidas).

---

**Fin de la Unidad 2.** Con estos tres notebooks recorriste el ciclo completo: manejar errores de forma prolija, organizar el código en módulos/paquetes con entornos aislados, y persistir datos en distintos formatos — la base de todo lo que construiste en las Unidades 3, 4, 5 y 6.